# Independence Tests Demo

This notebook shows examples of linear, nonlinear, nonmonotonic, and independent relationships, and tests them using various statistical measures.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import scale
import dcor
from minepy import MINE
import pandas as pd

np.random.seed(42)


In [ ]:

def linear_data(n=500):
    x = np.random.rand(n)
    y = 3 * x + np.random.normal(0, 0.1, n)
    return x, y

def nonlinear_data(n=500):
    x = np.random.uniform(0, 2 * np.pi, n)
    y = np.sin(x) + np.random.normal(0, 0.1, n)
    return x, y

def nonmonotonic_data(n=500):
    x = np.random.rand(n)
    y = np.sin(10 * np.pi * x) + np.random.normal(0, 0.1, n)
    return x, y

def independent_data(n=500):
    x = np.random.rand(n)
    y = np.random.rand(n)
    return x, y


In [ ]:

def run_tests(x, y):
    x_scaled = scale(x.reshape(-1, 1))
    y_scaled = scale(y.reshape(-1, 1))
    results = {}
    results["Pearson r"] = pearsonr(x, y)[0]
    results["Spearman ρ"] = spearmanr(x, y)[0]
    mi = mutual_info_regression(x_scaled, y_scaled.ravel(), discrete_features=False)
    results["Mutual Info"] = mi[0]
    results["Distance Corr"] = dcor.distance_correlation(x, y)
    mine = MINE(alpha=0.6, c=15)
    mine.compute_score(x, y)
    results["MIC"] = mine.mic()
    return results


In [ ]:

generators = {
    "Linear": linear_data,
    "Nonlinear": nonlinear_data,
    "Nonmonotonic": nonmonotonic_data,
    "Independent": independent_data
}

all_results = []

for name, generator in generators.items():
    x, y = generator()
    result = run_tests(x, y)
    result["Example"] = name
    all_results.append(result)

df = pd.DataFrame(all_results)
df.set_index("Example", inplace=True)
df.round(3)


In [ ]:

def plot_examples():
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    titles = list(generators.keys())
    for ax, title in zip(axes.ravel(), titles):
        x, y = generators[title]()
        ax.scatter(x, y, alpha=0.6)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_examples()
